In [79]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import shapiro, levene
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

pd.options.mode.chained_assignment = None
warnings.filterwarnings('ignore')

In [80]:
df = pd.read_csv('source/FReDA4.csv')
# df3 = pd.read_csv('source/FReDA3.csv')
df["Group4"] = None

In [81]:
# Satisfied
df.loc[df['Group3'] == 'Couple Satisfaction', 'Group4'] = 'Satisfied'

# Deprived groups
df.loc[df['Group1'] == 'SubGroup-DeprivedBoth', 'Group4'] = 'Deprived_Both'
df.loc[df['Group1'] == 'SubGroup-DeprivedMe', 'Group4'] = 'Deprived_Me'
df.loc[df['Group1'] == 'SubGroup-DeprivedPartner', 'Group4'] = 'Deprived_Partner'
#
# Oversaturated groups
df.loc[df['Group1'] == 'SubGroup-OversaturatedBoth', 'Group4'] = 'Oversaturated_Both'
df.loc[df['Group1'] == 'SubGroup-OversaturatedMe', 'Group4'] = 'Oversaturated_Me'
df.loc[df['Group1'] == 'SubGroup-OversaturatedPartner', 'Group4'] = 'Oversaturated_Partner'

df.loc[df['Group3'] == 'Couple Mixed', 'Group4'] = 'Mixed'

In [82]:
unique_values = df['Group4'].value_counts() #Subgroups nested on Group3
print(unique_values)

Group4
Satisfied                3842
Deprived_Both            3402
Deprived_Partner         2856
Deprived_Me              2084
Mixed                     660
Oversaturated_Partner     382
Oversaturated_Me          284
Oversaturated_Both         90
Name: count, dtype: int64


In [83]:
unique_values = df['Group3'].value_counts() #Groups
print(unique_values)

Group3
Couple Deprivation       8342
Couple Satisfaction      3842
Couple Oversaturation     756
Couple Mixed              660
Name: count, dtype: int64


In [84]:
df = df.rename(columns={
    'Self-esteem': 'Self_esteem',
    'Life Satisfaction': 'Life_Satisfaction',
    'Communication Quality': 'Communication_Quality',
    'Relationship Satisfaction': 'Relationship_Satisfaction',
    'Conflict Management': 'Conflict_Management'
})

In [85]:
traits = [
    'Neuroticism',
    'Extraversion',
    'Openness',
    'Agreeableness',
    'Conscientiousness',

    'Depressiveness',
    'Loneliness',
    'Self_esteem',
    'Life_Satisfaction',
    'Health',

    'Relationship_Satisfaction',
    'Communication_Quality',
    'Conflict_Management'
]

In [86]:
df2 = df.dropna(subset=traits).copy()

In [87]:
unique_values = df2['Group4'].value_counts() #Subgroups nested on Group3
print(unique_values)

Group4
Satisfied                3335
Deprived_Both            3026
Deprived_Partner         2506
Deprived_Me              1857
Mixed                     559
Oversaturated_Partner     331
Oversaturated_Me          240
Oversaturated_Both         74
Name: count, dtype: int64


In [88]:
unique_values = df2['Group3'].value_counts() #Groups
print(unique_values)

Group3
Couple Deprivation       7389
Couple Satisfaction      3335
Couple Oversaturation     645
Couple Mixed              559
Name: count, dtype: int64


In [89]:
import statsmodels.formula.api as smf
dv = 'Openness'
formula = "Neuroticism ~ C(Group3, Treatment('Couple Satisfaction'))"
model = smf.mixedlm(formula, data=df2, groups=df2['CoupleId']).fit()
print(model.summary())

# model = smf.mixedlm(dv +' ~ C(Group3)', data=df2, groups=df2["CoupleId"])
# result = model.fit()
# print(result.summary())

                                     Mixed Linear Model Regression Results
Model:                              MixedLM                   Dependent Variable:                   Neuroticism
No. Observations:                   11928                     Method:                               REML       
No. Groups:                         6615                      Scale:                                5.6672     
Min. group size:                    1                         Log-Likelihood:                       -27276.5845
Max. group size:                    2                         Converged:                            Yes        
Mean group size:                    1.8                                                                        
---------------------------------------------------------------------------------------------------------------
                                                                     Coef. Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

df2['Partner_Role'] = df2['Role'].map({'Partner': 'Anchor', 'Anchor': 'Partner'})

# 2. Merge the dataframe with itself on Couple_ID and the swapped Roles
df3 = pd.merge(
    df2,
    df2[['CoupleId', 'Role', 'Desire']],
    left_on=['CoupleId', 'Partner_Role'],
    right_on=['CoupleId', 'Role'],
    suffixes=('_Own', '_Partner')
)

# Clean up redundant columns from the merge
df3 = df3.drop(columns=['Role_Partner', 'Partner_Role']).rename(columns={'Role_Own': 'Role', 'Desire_Own': 'Desire_Actor'})

print(df3[['CoupleId', 'Role', 'Desire_Actor', 'Desire_Partner', 'Neuroticism']])

In [100]:

df3['Desire_Actor_C'] = df3['Desire_Actor'] - df3['Desire_Actor'].mean()
df3['Desire_Partner_C'] = df3['Desire_Partner'] - df3['Desire_Partner'].mean()

# 2. Define the distinguishable APIM formula
# This calculates unique Actor and Partner effects for each Role group
formula = "Neuroticism ~ Role + Desire_Actor_C:Role + Desire_Partner_C:Role"

# 3. Fit the model clustering by the Couple_ID
model = smf.mixedlm(formula, data=df3, groups=df3['CoupleId']).fit()

print(model.summary())
"""
"To evaluate the actor and partner effects of touch desire on individual Neuroticism across distinguishable dyadic roles (Anchor vs. Partner), a Linear Mixed Model with a random intercept for couple identity was performed. Predictor variables were grand-mean centered prior to analysis.The baseline Neuroticism score for Anchors at mean levels of touch desire was 7.89 ($SE = 0.033, p < .001$), which did not significantly differ from the baseline score of Partners ($\beta = 0.09, SE = 0.047, p = .063$). Results indicated that neither the actor effects nor the partner effects of touch desire were statistically significant for either Anchors or Partners (all $p > .25$). Furthermore, the random intercept variance for couple identity was estimated at 0.000, indicating that individual variance in Neuroticism was entirely independent of couple clustering.
"""

                  Mixed Linear Model Regression Results
Model:                   MixedLM      Dependent Variable:      Neuroticism
No. Observations:        10626        Method:                  REML       
No. Groups:              5313         Scale:                   5.7546     
Min. group size:         2            Log-Likelihood:          -24387.8634
Max. group size:         2            Converged:               Yes        
Mean group size:         2.0                                              
--------------------------------------------------------------------------
                               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------
Intercept                       7.891    0.033 239.761 0.000  7.827  7.956
Role[T.Partner]                 0.086    0.047   1.858 0.063 -0.005  0.178
Desire_Actor_C:Role[Anchor]    -0.018    0.030  -0.600 0.549 -0.077  0.041
Desire_Actor_C:Role[Partner]    0.016    0.0